In [13]:
import pickle
import os
from tqdm import tqdm 
from datasets import load_dataset
import pickle
import numpy as np
from datasets import Dataset, DatasetDict

In [14]:
FULL_MERGE_AVAILABLE = False

## Patch for Huggingface Dataset

This adds our review results to the huggingface dataset "LabHC/bias_in_bios" found at https://huggingface.co/datasets/LabHC/bias_in_bios.

In [17]:
if not FULL_MERGE_AVAILABLE:
    dataset = load_dataset("LabHC/bias_in_bios")

    # labels in the order specified here: https://huggingface.co/datasets/LabHC/bias_in_bios
    # not included in the datasets metadata
    huggingface_label = ['accountant', 'architect', 'attorney', 'chiropractor', 'comedian', 'composer', 'dentist', 'dietitian', 'dj', 'filmmaker', 'interior_designer', 'journalist', 'model', 'nurse', 'painter', 'paralegal', 'pastor', 'personal_trainer', 'photographer', 'physician', 'poet', 'professor', 'psychologist', 'rapper', 'software_engineer', 'surgeon', 'teacher', 'yoga_teacher']
    partitions = ['train', 'test', 'dev']

    dataset['train'].info

    # load the additional info
    with open('data/huggingface_patch.pkl', 'rb') as handle:
        review_data = pickle.load(handle)

In [18]:
review_data['train'][0]

{'text_duplicates': [],
 'titles_supervised': nan,
 'raw_titles_supervised': nan,
 'gender_supervised': 'M',
 'auto_title': 'professor',
 'auto_raw_title': 'associate professor',
 'review': 0.0,
 'valid': -1.0,
 'style_valid': -1.0,
 'duplicates': []}

In [19]:
if not FULL_MERGE_AVAILABLE:
    # merge data
    merged_dataset = {part: [] for part in partitions}
    dup_id_list = []

    for part in partitions:
        print(part)
        for i, sample in tqdm(enumerate(dataset[part])):
            text = sample['hard_text']
            patch = review_data[part][i]
            new_sample = sample.copy()

            if patch['review'] == 1:
                sample['gender'] = patch['gender_supervised']
            new_sample.update({'titles_supervised': patch['titles_supervised'], 'review': patch['review'], 'valid': patch['valid']})
            merged_dataset[part].append(new_sample)

train


257478it [00:05, 47224.58it/s]


test


99069it [00:02, 47866.37it/s]


dev


39642it [00:00, 50329.21it/s]


## Load complete merged dataset if available

In [20]:
if FULL_MERGE_AVAILABLE:
    with open('data/huggingface_merge_complete.pkl', 'rb') as handle:
        review_data = pickle.load(handle)

In [21]:
review_data['train'][0]

{'text_duplicates': [],
 'titles_supervised': nan,
 'raw_titles_supervised': nan,
 'gender_supervised': 'M',
 'auto_title': 'professor',
 'auto_raw_title': 'associate professor',
 'review': 0.0,
 'valid': -1.0,
 'style_valid': -1.0,
 'duplicates': []}

## Filter the dataset and convert to huggingface dataset

In [44]:
import pickle
import os
from tqdm import tqdm 
from datasets import load_dataset
import pickle
import numpy as np
from datasets import Dataset, DatasetDict

# select the fields to keep from the dataset (if non-reviewed samples are used, some fields may not be set for those not included in the review!)
KEYS_TO_COPY = ['hard_text', 'profession', 'gender', 'raw', 'titles_supervised', 'review', 'valid', 'name']
CLASSES = ['architect', 'surgeon', 'dentist', 'teacher', 'psychologist', 'nurse', 'photographer', 'physician', 'attorney', 'journalist', 'model', 'software engineer']
#CLASSES_COMPLETE =['architect', 'surgeon', 'dentist', 'teacher', 'psychologist', 'nurse', 'photographer', 'physician', 'attorney', 'journalist', '*writer', 'yoga teacher', '*editor', '*trainer', 'poet', '*artist/ designer', 'professor', '*entrepeneur', 'personal trainer', '*director/producer', 'dietitian', '*software architect', 'software engineer', 'filmmaker', '*researcher', '*consultant/coaches']

with open('data/huggingface_merge_complete.pkl', 'rb') as handle:
    merged_dataset = pickle.load(handle)

# filter the dataset: Copies the keys specified above and select a subset of classes (from the review). If non-reviewed samples are kept (and classes not part of the review) 
# there may be samples with empty fields (e.g. supervised titles). Name and raw text are only available if the full merged dataset (not the patch) is available.
#   classes: a (sub)set of classes to keep (should be part of the review)
#   single label: only keep those samples with a single label (otherwise get one-hot encoded labels)
#   review only: only those samples that were part of the review
#   valid only: only those considered valid in the review
#   remove_other_classes: whether to remove samples without labels (in the classes selection); if False and single_label=True these unlabeled samples would get a -1 label
def filter_dataset(dataset: dict, classes: list, single_label=True, review_only=True, valid_only=True, remove_other_classes=True):

    splits = dataset.keys()
    split_dict = {}
    filtered_dataset = {split: [] for split in splits}
    for split in splits:
        for elem in dataset[split]:
            if valid_only and elem['valid'] != 1:
                continue
            if review_only and elem['review'] != 1:
                continue
            sel_titles = [title for title in elem['titles_supervised'] if title in classes]
            if single_label and len(sel_titles) > 1:
                continue
                
            if remove_other_classes and len(sel_titles) == 0:
                continue
                
            new_entry = {k: elem[k] for k in KEYS_TO_COPY}
            if single_label:
                label = -1 if len(sel_titles) == 0 else classes.index(sel_titles[0])
            else: # multi-label / one-hot encoded
                label = np.zeros(len(classes))
                for title in sel_titles:
                    label[classes.index(title)] = 1
            new_entry.update({'label': label, 'auto_title': elem['auto_title']})
            filtered_dataset[split].append(new_entry)
        print(split, len(filtered_dataset[split]))
        
        cur_split = {k: [elem[k] for elem in filtered_dataset[split]] for k in filtered_dataset[split][0].keys()}
        split_dict[split] = Dataset.from_dict(cur_split, split=split)
    return DatasetDict(split_dict)

In [45]:
# only valid samples, multi-label
fdata = filter_dataset(merged_dataset, CLASSES, single_label=False, review_only=True, valid_only=True, remove_other_classes=False)

train 8555
test 3099
dev 1276


In [46]:
# all reviewed data
fdata = filter_dataset(merged_dataset, CLASSES, single_label=False, review_only=True, valid_only=False, remove_other_classes=False)

train 13373
test 4977
dev 2011


In [29]:
# all reviewed samples, single-label
fdata = filter_dataset(merged_dataset, CLASSES, True, True, False)

train 5494
test 1965
dev 798


In [39]:
def get_split_data(ds, split):
    X = ds[split]['hard_text']
    y = np.array(ds[split]['label'])
    gender = np.array(ds[split]['gender'])

    return X, y, gender

text_train, y_train, gender_train = get_split_data(fdata, 'train')
text_dev, y_dev, gender_dev = get_split_data(fdata, 'dev')
text_test, y_test, gender_test = get_split_data(fdata, 'test')

In [40]:
y_test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(4977, 10))

In [51]:
y = fdata['dev']['label']
np.sum(y,axis=0)

array([ 26.,  90.,  15.,  89.,  68.,  41., 170.,  22., 144., 133.])

In [ ]:
dataset